In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

SILVER_TABLES = [
    f"{SILVER_SCHEMA}.match_results_football_data",
    f"{SILVER_SCHEMA}.football_data_org_matches",
]

GOLD_TABLE = f"{GOLD_SCHEMA}.dim_date"


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

BUSINESS_KEY = [
    "date_key",
]

REQUIRED_SILVER_COLUMNS = [
    "match_date",
]

REQUIRED_GOLD_COLUMNS = [
    "date_key",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "year_month",
    "day_of_month",
    "day_of_week",
    "day_name",
    "week_of_year",
    "is_weekend",
]


# ---------------------------------------------------------------------------
# Validate source tables
# ---------------------------------------------------------------------------

source_date_ranges = []

for table_name in SILVER_TABLES:

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

    silver_df = spark.table(table_name)

    validate_columns_exist(
        silver_df,
        REQUIRED_SILVER_COLUMNS,
    )

    source_metrics = validate_required_not_null(
        silver_df,
        REQUIRED_SILVER_COLUMNS,
        fail_on_null=True,
    )

    if source_metrics["row_count"] == 0:
        raise ValueError(
            f"Required Silver table is empty: {table_name}"
        )

    dates_df = silver_df.select(
        F.to_date("match_date").alias("full_date")
    )

    date_metrics = dates_df.agg(
        F.min("full_date").alias("min_date"),
        F.max("full_date").alias("max_date"),
        F.count("full_date").alias("valid_dates"),
    ).first()

    if date_metrics["valid_dates"] != source_metrics["row_count"]:
        raise ValueError(
            f"Invalid match dates found in {table_name}."
        )

    source_date_ranges.append({
        "table_name": table_name,
        "min_date": date_metrics["min_date"],
        "max_date": date_metrics["max_date"],
    })

    print(
        f"Source validated: {table_name}, "
        f"rows={source_metrics['row_count']}, "
        f"min_date={date_metrics['min_date']}, "
        f"max_date={date_metrics['max_date']}"
    )


# ---------------------------------------------------------------------------
# Determine calendar date range
# ---------------------------------------------------------------------------

# Generate the calendar using actual match dates.
# Both historical matches and future API fixtures are included.

start_date = min(
    row["min_date"]
    for row in source_date_ranges
)

end_date = max(
    row["max_date"]
    for row in source_date_ranges
)

if start_date > end_date:
    raise ValueError(
        "Invalid calendar range: "
        "start_date is greater than end_date."
    )

# Guard against unexpected source dates that could generate
# an excessively large calendar dimension.

if start_date.year < 1900 or end_date.year > 2100:
    raise ValueError(
        "Calendar date range is outside supported boundaries: "
        f"{start_date} to {end_date}"
    )

expected_days = (
    end_date - start_date
).days + 1

if expected_days > 36525:
    raise ValueError(
        "Calendar range exceeds 100 years. "
        "Verify source match dates."
    )

print(
    f"Calendar range: {start_date} to {end_date}"
)

print(
    f"Expected calendar days: {expected_days}"
)


# ---------------------------------------------------------------------------
# Generate continuous calendar
# ---------------------------------------------------------------------------

# Spark range generates one row for every calendar day.
# This avoids collecting individual dates to the driver.
#
# The full_date column is generated deterministically
# from the earliest date and its day offset.

calendar_df = (
    spark.range(expected_days)
    .select(
        F.date_add(
            F.lit(start_date).cast("date"),
            F.col("id").cast("int"),
        ).alias("full_date")
    )
)


# ---------------------------------------------------------------------------
# Build Gold date dimension
# ---------------------------------------------------------------------------

# The date key uses YYYYMMDD format.
# ISO day-of-week numbering is used: Monday=1, Sunday=7.
# Month and day names are explicitly formatted in English.

gold_df = (
    calendar_df
    .withColumn(
        "date_key",
        F.date_format(
            "full_date",
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "year",
        F.year("full_date"),
    )
    .withColumn(
        "quarter",
        F.quarter("full_date"),
    )
    .withColumn(
        "month",
        F.month("full_date"),
    )
    .withColumn(
        "month_name",
        F.element_at(
            F.array(
                *[
                    F.lit(name)
                    for name in [
                        "January",
                        "February",
                        "March",
                        "April",
                        "May",
                        "June",
                        "July",
                        "August",
                        "September",
                        "October",
                        "November",
                        "December",
                    ]
                ]
            ),
            F.month("full_date"),
        ),
    )
    .withColumn(
        "year_month",
        F.date_format(
            "full_date",
            "yyyy-MM",
        ),
    )
    .withColumn(
        "day_of_month",
        F.dayofmonth("full_date"),
    )
    .withColumn(
        "day_of_week",
        F.weekday("full_date") + F.lit(1),
    )
    .withColumn(
        "day_name",
        F.element_at(
            F.array(
                *[
                    F.lit(name)
                    for name in [
                        "Monday",
                        "Tuesday",
                        "Wednesday",
                        "Thursday",
                        "Friday",
                        "Saturday",
                        "Sunday",
                    ]
                ]
            ),
            F.weekday("full_date") + F.lit(1),
        ),
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("full_date"),
    )
    .withColumn(
        "is_weekend",
        F.col("day_of_week").isin(6, 7),
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "year_month",
        "day_of_month",
        "day_of_week",
        "day_name",
        "week_of_year",
        "is_weekend",
    )
)


# ---------------------------------------------------------------------------
# Validate Gold date dimension
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

# Calendar must contain exactly one row per day.

if gold_count != expected_days:
    raise ValueError(
        "Gold calendar row count mismatch: "
        f"expected={expected_days}, actual={gold_count}"
    )

# Every date key must be unique.

duplicate_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    display(duplicate_keys)

    raise ValueError(
        "Duplicate date_key values found in dim_date."
    )

# Verify date keys follow the YYYYMMDD convention.

invalid_date_keys = (
    gold_df
    .filter(
        F.col("date_key")
        != F.date_format(
            "full_date",
            "yyyyMMdd",
        ).cast("int")
    )
    .limit(1)
    .count()
)

if invalid_date_keys > 0:
    raise ValueError(
        "Invalid date_key values found in dim_date."
    )

# Verify calendar attributes.

invalid_attributes = (
    gold_df
    .filter(
        (~F.col("month").between(1, 12))
        | (~F.col("quarter").between(1, 4))
        | (~F.col("day_of_week").between(1, 7))
        | (
            F.col("is_weekend")
            != F.col("day_of_week").isin(6, 7)
        )
    )
    .limit(1)
    .count()
)

if invalid_attributes > 0:
    raise ValueError(
        "Invalid calendar attributes found in dim_date."
    )

# Validate calendar boundaries and continuity.

calendar_metrics = gold_df.agg(
    F.min("full_date").alias("min_date"),
    F.max("full_date").alias("max_date"),
    F.countDistinct("full_date").alias("distinct_dates"),
).first()

if (
    calendar_metrics["min_date"] != start_date
    or calendar_metrics["max_date"] != end_date
    or calendar_metrics["distinct_dates"] != expected_days
):
    raise ValueError(
        "Gold calendar is not continuous or "
        "does not match source date boundaries."
    )

print(
    "Gold validation successful: "
    f"{gold_count} continuous calendar dates."
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for this small dimension.
# Dates and calendar attributes are generated deterministically.
# Rerunning the notebook does not append duplicate dates.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_metrics = validate_required_not_null(
    saved_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

saved_count = saved_metrics["row_count"]

if saved_count != expected_days:
    raise ValueError(
        "Post-write validation failed: "
        f"expected {expected_days} dates, "
        f"found {saved_count}."
    )

saved_duplicate_keys = (
    saved_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: "
        "duplicate date_key values."
    )

saved_calendar_metrics = saved_df.agg(
    F.min("full_date").alias("min_date"),
    F.max("full_date").alias("max_date"),
    F.countDistinct("full_date").alias("distinct_dates"),
).first()

if (
    saved_calendar_metrics["min_date"] != start_date
    or saved_calendar_metrics["max_date"] != end_date
    or saved_calendar_metrics["distinct_dates"] != expected_days
):
    raise ValueError(
        "Post-write validation failed: "
        "calendar coverage or continuity is incorrect."
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD DATE DIMENSION LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Total calendar dates: {saved_count}")
print(f"Earliest date: {start_date}")
print(f"Latest date: {end_date}")
print(f"Duplicate date keys: {saved_duplicate_keys}")
print("=" * 55)

display(
    saved_df.orderBy("full_date")
)
